In [24]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import numpy as np
from iminuit import Minuit
from iminuit.cost import LeastSquares
from scipy.stats import chi2


In [25]:
def line( x , a , b):
    return a*x +b

# Plateaux


In [26]:
#prendo path per tutti uguale
DATA_FILE = Path('Data') / 'ch_4-5.csv'
if not DATA_FILE.exists():
    for p in Path.cwd().parents:
        candidate = p / 'Data' / 'ch_4-5.csv'
        if candidate.exists():
            DATA_FILE = candidate
            break
df_varia = pd.read_csv(DATA_FILE, sep = r',', skiprows = 0, engine='python')
display(df_varia)

,p_tensione[V],g_tensione[V],p_threshold[mV],g_threshold[mV],p_count,g_count,pg_doppie,gm_doppie,pgm_triple,m_tensione[V],m_threshold[mV]
0,780,950,141,120,1420,1548.0,683.0,NaN,NaN,NaN,NaN
1,780,950,253,153,128,815.0,51.0,NaN,NaN,NaN,NaN
2,780,950,200,183,418,466.0,131.0,NaN,NaN,NaN,NaN
3,780,950,151,209,1117,269.0,133.0,NaN,NaN,NaN,NaN
4,780,950,234,192,218,372.0,65.0,NaN,NaN,NaN,NaN
5,780,950,212,201,378,334.0,106.0,NaN,NaN,NaN,NaN
6,780,950,238,224,192,197.0,37.0,NaN,NaN,NaN,NaN
7,780,950,187,171,563,518.0,176.0,NaN,NaN,NaN,NaN
8,800,950,237,167,373,557.0,126.0,NaN,NaN,NaN,NaN
9,800,980,187,173,1088,1115.0,466.0,NaN,NaN,NaN,NaN


In [27]:
df_880p = df_varia[df_varia['p_tensione[V]'] == 880]
df_1050g = df_varia[df_varia['g_tensione[V]'] == 1050]
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_880p['p_threshold[mV]'], y=df_880p['p_count'], mode='markers', name='p_threshold[mV] = 880', error_y=dict(type='data', array=np.sqrt(df_880p['p_count']), visible=True)))
fig.add_trace(go.Scatter(x=df_1050g['g_threshold[mV]'], y=df_1050g['g_count'], mode='markers', name='g_threshold[mV] = 1050', error_y=dict(type='data', array=np.sqrt(df_1050g['g_count']), visible=True)))

In [28]:
df_880p = df_varia[(df_varia['p_tensione[V]'] == 880) & (df_varia["p_threshold[mV]"]>140) & (df_varia["p_threshold[mV]"]<180)]




p_x = df_880p['p_threshold[mV]'].to_numpy()
p_y = df_880p['p_count'].to_numpy()
p_yerr = np.sqrt(df_880p['p_count'].to_numpy())

# sort p arrays by p_x while keeping pairings
idx = np.argsort(p_x)
p_x = p_x[idx]
p_y = p_y[idx]
p_yerr = p_yerr[idx]

p_LS = LeastSquares( p_x , p_y , p_yerr , line)

m = Minuit( p_LS , a = 0 , b = 1e3)

#m.fixed["a"] = True

m.migrad()
m.hesse()

print( m.fval)
display(m)

2.276672021718203


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2.277 (χ²/ndof = 1.1)      │              Nfcn = 50               │
│ EDM = 3e-18 (Goal: 0.0002)       │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │    -18    │     4     │            │            │         │         │       │
│ 1 │ b    │   8.4e3   │   0.6e3   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬───────────────────┐
│   │        a        b │
├───┼───────────────────┤
│ a │     13.9 -2.259e3 │
│ b │ -2.259e3 3.67e+05 │
└───┴───────────────────┘

In [29]:
df_1050g = df_varia[(df_varia['g_tensione[V]'] == 1050) & (df_varia["g_threshold[mV]"]>110) & (df_varia["g_threshold[mV]"]<130)]

g_x = df_1050g['g_threshold[mV]'].to_numpy()
g_y = df_1050g['g_count'].to_numpy()
g_yerr = np.sqrt(df_1050g['g_count'].to_numpy())

# sort g arrays by g_x while keeping pairings
idx = np.argsort(g_x)
g_x = g_x[idx]
g_y = g_y[idx]
g_yerr = g_yerr[idx]

g_LS = LeastSquares( g_x , g_y , g_yerr , line)

m = Minuit( g_LS , a = 0 , b = 1e3)

#m.fixed["a"] = True

m.migrad()
m.hesse()

display(m)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4.147e-21                  │              Nfcn = 55               │
│ EDM = 4.13e-21 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │    -6     │    12     │            │            │         │         │       │
│ 1 │ b    │   6.2e3   │   1.4e3   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬───────────────────┐
│   │        a        b │
├───┼───────────────────┤
│ a │      137 -16.29e3 │
│ b │ -16.29e3 1.93e+06 │
└───┴───────────────────┘

# Loop

## Partenope

In [33]:
n_punti = 3

df_880p = df_varia[(df_varia['p_tensione[V]'] == 880)]


p_x = df_880p['p_threshold[mV]'].to_numpy()
p_y = df_880p['p_count'].to_numpy()
p_yerr = np.sqrt(df_880p['p_count'].to_numpy())

print( len(p_x))

idx = np.argsort(p_x)
p_x = p_x[idx]
p_y = p_y[idx]
p_yerr = p_yerr[idx]


for i in range(len(p_x)-n_punti+1):
    x = p_x[i:i+n_punti]
    y = p_y[i:i+n_punti]
    err = p_yerr[i:i+n_punti]

    loop_LS = LeastSquares( x , y , err , line)

    m = Minuit( loop_LS , a = -0.1 , b = 1e3)
    m.limits["a"] = (None , 0)
    #m.fixed["a"] = True
    m.migrad()
    m.hesse()
    chi_square_prob = 1 - chi2.cdf(m.fval, df=m.ndof)
    print( i , m.values["a"] , m.fval/m.ndof , chi_square_prob)





18
0 -982.9196801708614 1889.0432060197154 0.0
1 -282.05539691489093 40.362203016838606 2.1098356395299334e-10
2 -133.61914142540334 26.87540854372685 2.1700253860501562e-07
3 -66.34840114848613 1.6364505664608349 0.20081316098522506
4 -40.19527470212968 4.293400042545715 0.038260579613297985
5 -23.010034332082544 1.4612112704626088 0.2267374932679449
6 -16.745416272746056 0.3890417743704246 0.532803516249693
7 -12.933892590662566 1.1619288936189447 0.28106586381703114
8 -0.035574694666733464 10.80235102221296 0.0010137127395168921
9 -16.5561685438348 2.1495630201234084 0.14261047361552992
10 -17.4516861504012 2.5551533580760335 0.10993514625764822
11 -104.25271087071991 73.87668074759974 0.0
12 -92.61021570069185 575.2918415660569 0.0
13 -92.86095144764991 574.849714710783 0.0
14 -65.01395773544137 707.5116918225614 0.0
15 -0.00018258404872129042 62.687752303648466 2.4424906541753444e-15


In [36]:
i = 8

x = p_x[i:i+n_punti]
y = p_y[i:i+n_punti]
err = p_yerr[i:i+n_punti]

loop_LS = LeastSquares( x , y , err , line)

m = Minuit( loop_LS , a = -0.1 , b = 1e3)

#m.limits["a"] = (None , 0)
#m.fixed["a"] = True
m.migrad()
m.hesse()

display(m)
print( i , m.values["a"] , m.fval/m.ndof)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2.276 (χ²/ndof = 2.3)      │              Nfcn = 57               │
│ EDM = 3.09e-17 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │    -18    │     6     │            │            │         │         │       │
│ 1 │ b    │   8.4e3   │   1.0e3   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬───────────────────┐
│   │        a        b │
├───┼───────────────────┤
│ a │     37.2  -6.20e3 │
│ b │  -6.20e3 1.03e+06 │
└───┴───────────────────┘

8 -17.866546266021455 2.276374753247251


## Giunone

In [43]:
n_punti = 3

df_1050g = df_varia[(df_varia['g_tensione[V]'] == 1050)]

g_x = df_1050g['g_threshold[mV]'].to_numpy()
g_y = df_1050g['g_count'].to_numpy()
g_yerr = np.sqrt(df_1050g['g_count'].to_numpy())

# sort g arrays by g_x while keeping pairings
idx = np.argsort(g_x)
g_x = g_x[idx]
g_y = g_y[idx]
g_yerr = g_yerr[idx]


for i in range(len(g_x)-n_punti+1):
    x = g_x[i:i+n_punti]
    y = g_y[i:i+n_punti]
    err = g_yerr[i:i+n_punti]

    loop_LS = LeastSquares( x , y , err , line)

    m = Minuit( loop_LS , a = 0 , b = 1e3)
    #m.limits["a"] = (None , 0)
    m.fixed["a"] = True
    m.migrad()
    m.hesse()
    chi_square_prob = 1 - chi2.cdf(m.fval, df=m.ndof)
    print( i , m.values["a"] , m.fval/m.ndof , chi_square_prob)




0 0.0 2857.1060742468476 0.0
1 0.0 375.5691364511896 0.0
2 0.0 47.369186570173824 0.0
3 0.0 24.683085711406463 1.9066526135702588e-11
4 0.0 10.788823309129738 2.0628781251752493e-05
5 0.0 10.848959765317375 1.9424803738399277e-05
6 0.0 23.329634882632185 7.380207556195728e-11
7 0.0 20.345816678427784 1.4585592733595831e-09
8 0.0 22.66097820576553 1.4403311876520775e-10
9 0.0 73.21224155530015 0.0
10 0.0 95.31387836740166 0.0
11 0.0 83.91629698475398 0.0
12 0.0 196.2641862761903 0.0
13 0.0 113.96028451546513 0.0


In [ ]:
i = 5

x = g_x[i:i+n_punti]
y = g_y[i:i+n_punti]
err = g_yerr[i:i+n_punti]

loop_LS = LeastSquares( x , y , err , line)

m = Minuit( loop_LS , a = 0 , b = 1e3)

#m.limits["a"] = (None , 0)
m.fixed["a"] = True
m.migrad()
m.hesse()

display(m)
print( i , m.values["a"] , m.fval/m.ndof)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2.448 (χ²/ndof = 1.2)      │              Nfcn = 45               │
│ EDM = 7.49e-19 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   -23.3   │    2.8    │            │            │         │         │       │
│ 1 │ b    │   8.3e3   │   0.4e3   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬───────────────────┐
│   │        a        b │
├───┼───────────────────┤
│ a │     7.83 -1.026e3 │
│ b │ -1.026e3 1.36e+05 │
└───┴───────────────────┘

5 -23.26640195891425 1.224167165238366
